In [2]:
!pip install torch numpy matplotlib tqdm

In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [5]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-31 12:54:07--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-07-31 12:54:07 (20.5 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [6]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
print(f"Total characters : {len(text):,}")
print(f"Unique characters: {len(set(text))}")

Total characters : 1,115,394
Unique characters: 65


In [8]:
# Get all unique characters in the dataset
chars = sorted(list(set(text)))

# Vocabulary size
vocab_size = len(chars)

print(chars)
print(f"Vocabulary Size: {vocab_size}")

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocabulary Size: 65


In [9]:
# Character to Integer
stoi = {ch: i for i, ch in enumerate(chars)}

# Integer to Character
itos = {i: ch for i, ch in enumerate(chars)}

print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [10]:
def encode(s):
    """
    Convert a string into a list of token IDs.
    """
    return [stoi[c] for c in s]

In [11]:
encode("Hello")

[20, 43, 50, 50, 53]

In [12]:
def decode(tokens):
    """
    Convert token IDs back into a string.
    """
    return "".join([itos[i] for i in tokens])

In [13]:
sample = "Hello GPT!"

encoded = encode(sample)

decoded = decode(encoded)

print("Original :", sample)
print("Encoded  :", encoded)
print("Decoded  :", decoded)

Original : Hello GPT!
Encoded  : [20, 43, 50, 50, 53, 1, 19, 28, 32, 2]
Decoded  : Hello GPT!


In [14]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape)
print(data[:100])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [15]:
# Hyperparameters
batch_size = 64        # Number of sequences per batch
block_size = 128       # Context length (how many previous characters the model sees)

In [16]:
# 90% training, 10% validation
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Training tokens :", len(train_data))
print("Validation tokens:", len(val_data))

Training tokens : 1003854
Validation tokens: 111540


In [17]:
def get_batch(split):
    """
    Generate one batch of input-target pairs.
    """

    data_source = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(len(data_source) - block_size, (batch_size,))

    # Input sequences
    x = torch.stack([data_source[i:i+block_size] for i in ix])

    # Target sequences (shifted by one)
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)

In [18]:
x, y = get_batch("train")

print("Input Shape :", x.shape)
print("Target Shape:", y.shape)

Input Shape : torch.Size([64, 128])
Target Shape: torch.Size([64, 128])


In [19]:
print("Input Tokens:")
print(x[0])

print("\nDecoded Input:")
print(decode(x[0].tolist()))

print("\nTarget Tokens:")
print(y[0])

print("\nDecoded Target:")
print(decode(y[0].tolist()))

Input Tokens:
tensor([58,  8,  1, 20, 53, 61,  1, 52, 53, 61,  6,  1, 40, 53, 63,  2,  0,  0,
        25, 13, 25, 21, 24, 24, 21, 33, 31, 10,  0, 21,  1, 39, 51,  1, 50, 47,
        49, 43,  1, 63, 53, 59,  6,  1, 58, 46, 43, 63,  1, 57, 39, 63,  8,  0,
         0, 24, 17, 27, 26, 32, 17, 31, 10,  0, 35, 46, 63,  1, 58, 46, 39, 58,
         5, 57,  1, 57, 53, 51, 43,  1, 41, 53, 51, 44, 53, 56, 58,  8,  1, 35,
        46, 39, 58,  6,  1, 15, 39, 51, 47, 50, 50, 53,  1, 58, 46, 43, 56, 43,
        12,  0,  0, 15, 13, 25, 21, 24, 24, 27, 10,  0, 13, 63,  6,  1, 51, 63,
         1, 45], device='cuda:0')

Decoded Input:
t. How now, boy!

MAMILLIUS:
I am like you, they say.

LEONTES:
Why that's some comfort. What, Camillo there?

CAMILLO:
Ay, my g

Target Tokens:
tensor([ 8,  1, 20, 53, 61,  1, 52, 53, 61,  6,  1, 40, 53, 63,  2,  0,  0, 25,
        13, 25, 21, 24, 24, 21, 33, 31, 10,  0, 21,  1, 39, 51,  1, 50, 47, 49,
        43,  1, 63, 53, 59,  6,  1, 58, 46, 43, 63,  1, 57, 39, 63,  8,

In [20]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Each token directly predicts logits for the next token
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx shape: (B, T)
        logits = self.token_embedding_table(idx)   # (B, T, C)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, _ = self(idx)

            # Take only the last time step
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            next_token = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, next_token), dim=1)

        return idx

In [21]:
model = BigramLanguageModel(vocab_size)
model = model.to(device)

print(model)

BigramLanguageModel(
  (token_embedding_table): Embedding(65, 65)
)


In [22]:
xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Logits Shape :", logits.shape)
print("Loss :", loss.item())

Logits Shape : torch.Size([8192, 65])
Loss : 4.803889274597168


In [23]:
xb, yb = get_batch("train")

logits, loss = model(xb, yb)

print("Logits Shape :", logits.shape)
print("Loss :", loss.item())

Logits Shape : torch.Size([8192, 65])
Loss : 4.800065994262695


In [24]:
def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):

        logits, _ = self(idx)

        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

    return idx

In [25]:
def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):

        logits, _ = self(idx)

        logits = logits[:, -1, :]

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

    return idx

In [26]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(context, max_new_tokens=300)

print(decode(generated[0].tolist()))


H-n-; rvTQaqqF;YUyTuDTyFiZNJjR$W:xT!Fs-cL&KNQGJwzqTQYuL'xL'xJYbyEH:xPB.v
CB.
L.,kBAtUd3xXbdVkYZhbO;TkBUdmGo?PIEm$!ufqlS&gkO!:rtQSs?qDE,FiURkaUdVf-?k:ZTkGgZWQvC3KRWCd-AoA 3VtSh,Unp'xpwPw:rtCv
aYhlSSClxbco'wFNnQnGwQS'eAjfyVF;xpxMuRzeuTRJZGUdki?SvJB.kK
q
J$pCvAcjlpf?Z.JAoPApxfYL'?neEl?WbGLavsHS
:I3VEgQ


In [27]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_iters = 3000
for step in range(max_iters):

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if step % 300 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

Step 0 | Loss: 4.8409
Step 300 | Loss: 4.3931
Step 600 | Loss: 4.0105
Step 900 | Loss: 3.7309
Step 1200 | Loss: 3.4655
Step 1500 | Loss: 3.2720
Step 1800 | Loss: 3.0644
Step 2100 | Loss: 2.9430
Step 2400 | Loss: 2.8216
Step 2700 | Loss: 2.7774


In [28]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(context, max_new_tokens=500)

print(decode(generated[0].tolist()))


WhefoPOLMye

KdfoQYod
Hverxfrepr,qdVk'drpoousSs-WAR:3t BOPu y, pur'VSCXSo.Pes f be
gets NEYy, :
TENI h VIckTE:
O:ouryeats; hodPSG ouj--f cl hVOMa$nwim k th?
Whime.ist beGomowis&zCoke Y yoOPlexe ?Porizrainso tanqZAhme
LOf spTINGots.

NRendisevourunTindV:

yt trM$QUSuou h omK!Juri-heFo.
G nditlest migowy orathe ayourevest pof,z oun.
CIam nt;'ToHSP,
AN$Oneashow:vou oplVe he denan keme

AMy?!ZANKf e neny t liMQUdVMuiseto: nday mhth :
hsio TAnh iseif,
Womqus.

GHyzedond, methadavecrd, n stoxy?yot nTE


In [29]:
# Transformer Hyperparameters

n_embd = 384        # Embedding dimension
n_head = 6          # Number of attention heads
n_layer = 6         # Number of transformer blocks
dropout = 0.2

In [30]:
class Head(nn.Module):
    """
    One head of masked self-attention.
    """

    def __init__(self, head_size):
        super().__init__()

        # Linear projections
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Causal mask
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        # Compute Key, Query, Value
        k = self.key(x)
        q = self.query(x)

        # Attention scores
        wei = q @ k.transpose(-2, -1)

        # Scale
        wei = wei * (C ** -0.5)

        # Mask future tokens
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # Softmax
        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        # Values
        v = self.value(x)

        out = wei @ v

        return out

In [31]:
head_size = n_embd // n_head

head = Head(head_size).to(device)

x = torch.randn(batch_size, block_size, n_embd).to(device)

out = head(x)

print(out.shape)

torch.Size([64, 128, 64])


In [32]:
class MultiHeadAttention(nn.Module):
    """
    Multiple heads of self-attention running in parallel.
    """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(head_size * num_heads, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Concatenate outputs of all heads
        out = torch.cat([head(x) for head in self.heads], dim=-1)

        # Final projection
        out = self.proj(out)

        out = self.dropout(out)

        return out

In [33]:
multi_head = MultiHeadAttention(
    num_heads=n_head,
    head_size=n_embd // n_head
).to(device)

x = torch.randn(batch_size, block_size, n_embd).to(device)

out = multi_head(x)

print(out.shape)

torch.Size([64, 128, 384])


In [34]:
class FeedForward(nn.Module):
    """
    A simple feed-forward network.
    """

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [35]:
ff = FeedForward(n_embd).to(device)

x = torch.randn(batch_size, block_size, n_embd).to(device)

out = ff(x)

print(out.shape)

torch.Size([64, 128, 384])


In [36]:
class Block(nn.Module):
    """
    Transformer Block:
    Communication (Attention) followed by Computation (Feed Forward).
    """

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Multi-Head Attention with Residual Connection
        x = x + self.sa(self.ln1(x))

        # Feed Forward with Residual Connection
        x = x + self.ffwd(self.ln2(x))

        return x

In [37]:
block = Block(n_embd, n_head).to(device)

x = torch.randn(batch_size, block_size, n_embd).to(device)

out = block(x)

print(out.shape)

torch.Size([64, 128, 384])


In [38]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Token embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # Position embeddings
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Stack of Transformer Blocks
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        # Final LayerNorm
        self.ln_f = nn.LayerNorm(n_embd)

        # Output projection
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        # Combine token and position information
        x = tok_emb + pos_emb

        # Pass through Transformer Blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Vocabulary logits
        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Keep only the last block_size tokens
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)

            # Last token predictions
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [39]:
model = GPTLanguageModel().to(device)

xb, yb = get_batch("train")

xb = xb.to(device)
yb = yb.to(device)

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Logits shape: torch.Size([8192, 65])
Loss: 4.262516975402832


In [40]:
print(f"Model Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} Million")

Model Parameters: 10.74 Million


In [41]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [42]:
eval_iters = 200

In [43]:
@torch.no_grad()
def estimate_loss():
    out = {}

    model.eval()

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)

            X = X.to(device)
            Y = Y.to(device)

            _, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [45]:
max_iters = 5000
eval_interval = 500

for iter in range(max_iters):

    if iter % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {iter}: "
            f"Train Loss {losses['train']:.4f}, "
            f"Validation Loss {losses['val']:.4f}"
        )

    xb, yb = get_batch("train")

    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

Step 0: Train Loss 1.7242, Validation Loss 1.8672
Step 500: Train Loss 1.5192, Validation Loss 1.7099
Step 1000: Train Loss 1.4129, Validation Loss 1.6214
Step 1500: Train Loss 1.3449, Validation Loss 1.5761
Step 2000: Train Loss 1.2954, Validation Loss 1.5384
Step 2500: Train Loss 1.2614, Validation Loss 1.5269
Step 3000: Train Loss 1.2257, Validation Loss 1.5155
Step 3500: Train Loss 1.2006, Validation Loss 1.5072
Step 4000: Train Loss 1.1690, Validation Loss 1.5047
Step 4500: Train Loss 1.1447, Validation Loss 1.5009


In [46]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(context, max_new_tokens=500)

print(decode(generated[0].tolist()))


An which you sure can should not call much,
To one rest in order fine that I may know at
The queen, and in thing 'kissing her, bear with war;
The noble shame with deep rightful dulcontion
Ire in my sovereign, come to your grace,
And plant my great sleep; heself his name.

ISABELLA:
Will you hear me?

LUCIO:
Why?

ARCHBESMOPSON:
My hand I am was not more.

MERCUTIO:
I am going; let me hear a kind,
And a little fall to the duke. Nay, prove the trigod keeps
To quickly ground? what is our valour mai
